In [0]:
CATALOG = "airbnb_obs"
from pyspark.sql import functions as F

def diagnose_not_null(table_name, column, limit = 20):
    df = spark.table(f"{CATALOG}.bronze.{table_name}")
    return df.filter(F.col(column).isNull()).limit(limit)

def diagnose_accepted_values(table_name, column, allowed, limit= 20):
     df = spark.table(f"{CATALOG}.bronze.{table_name}")   
     return (df.filter(F.col(column).isNotNull() & ~F.col(column).isin(allowed)).limit(limit))

def diagnose_range(table_name, column, min, max, limit = 20):
    df = spark.table(f"{CATALOG}.bronze.{table_name}")  
    casted = F.col(column).cast("double")
    return df.filter((casted < min) | (casted > max) | (casted.isNull() & F.col(column).isNotNull())).limit(limit)

def diagnose_unique(table_name, column, limit = 20):
    df = spark.table(f"{CATALOG}.bronze.{table_name}")
    dupes = df.groupBy(column).count().filter(F.col("count") > 1)
    return df.join(dupes, on=column, how="inner").limit(limit)

print("diagnostic functios ready")

 

In [0]:
# CHECK 

print("=== Illegal booking_status values ===")
diagnose_accepted_values("bookings_broken", "booking_status",
                         ["confirmed", "cancelled"]).show()

print("=== Out-of-range nights_booked (valid 1–30) ===")
diagnose_range("bookings_broken", "nights_booked", 1, 30).show()

print("=== Null listing_id ===")
diagnose_not_null("bookings_broken", "listing_id").show()

In [0]:

from pyspark.sql import functions as F 
CATALOG = "airbnb_obs"  
df = spark.table(f"{CATALOG}.bronze.bookings_broken") 
df.filter(~F.col("booking_status").isin(["confirmed", "cancelled"])).select("booking_status").distinct().show() 